In [1]:
# CELL 1: Load datasets and prepare labels

import pandas as pd

majaz = pd.read_csv("majaz_dataset.csv")
non_majaz = pd.read_csv("non_majaz_random_sample.csv")

majaz["label"] = 1
non_majaz["label"] = 0

df = pd.concat([majaz, non_majaz], ignore_index=True)

TEXT_COL = "البيت_الكامل"
texts = df[TEXT_COL].astype(str).tolist()
y = df["label"].values

print(df.shape)
print(df["label"].value_counts())


(1684, 13)
label
1    842
0    842
Name: count, dtype: int64


In [3]:
# CELL A: Arabic tashkeel (diacritics) removal

import re

# Arabic diacritics Unicode range
ARABIC_DIACRITICS = re.compile(
    r"""
    ّ    | # Shadda
    َ    | # Fatha
    ً    | # Tanwin Fath
    ُ    | # Damma
    ٌ    | # Tanwin Damm
    ِ    | # Kasra
    ٍ    | # Tanwin Kasr
    ْ    | # Sukun
    ـ      # Tatweel
    """,
    re.VERBOSE
)

def remove_tashkeel(text: str) -> str:
    return re.sub(ARABIC_DIACRITICS, "", text)


In [4]:
# CELL B: Normalize verses (remove tashkeel BEFORE analysis)

texts_raw = df[TEXT_COL].astype(str).tolist()
texts = [remove_tashkeel(t) for t in texts_raw]

# quick sanity check
print(texts_raw[0])
print(texts[0])


مَرَرْتُ عَلى دارِ الحَبِيبِ فحَمْحَمَتْ جَوَادي وَهَلْ تَشْجو الجِيادَ المَعاهِدُ
مررت على دار الحبيب فحمحمت جوادي وهل تشجو الجياد المعاهد


In [5]:
# CELL 2: Initialize CAMeL Tools morphological analyzer

from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer
from camel_tools.tokenizers.word import simple_word_tokenize

db = MorphologyDB.builtin_db()
analyzer = Analyzer(db, backoff="NOAN_PROP")

# sanity check
print(analyzer.analyze("تشكو")[0])


{'diac': 'تَشْكُو', 'lex': 'شَكا', 'bw': 'تَ/IV2MS+شْكُو/IV', 'gloss': 'you_[masc.sg.]+complain;suffer', 'pos': 'verb', 'prc3': '0', 'prc2': '0', 'prc1': '0', 'prc0': '0', 'per': '2', 'asp': 'i', 'vox': 'a', 'mod': 'u', 'stt': 'na', 'cas': 'na', 'enc0': '0', 'rat': 'n', 'source': 'lex', 'form_gen': 'm', 'form_num': 's', 'd3seg': 'تَشْكُو', 'caphi': 't_a_sh_k_u_w', 'd1tok': 'تَشْكُو', 'd2tok': 'تَشْكُو', 'pos_logprob': -1.023208, 'd3tok': 'تَشْكُو', 'd2seg': 'تَشْكُو', 'pos_lex_logprob': -4.286608, 'num': 's', 'ud': 'VERB', 'gen': 'm', 'catib6': 'VRB', 'root': 'ش.ك.#', 'bwtok': 'تَ+_شْكُو', 'pattern': 'تَ1ْ2ُو', 'lex_logprob': -4.286608, 'atbtok': 'تَشْكُو', 'atbseg': 'تَشْكُو', 'd1seg': 'تَشْكُو', 'stem': 'شْكُو', 'stemgloss': 'complain;suffer', 'stemcat': 'IV_0hAnn'}


In [6]:
# CELL 3: Morpho-syntactic features

def morpho_features(text):
    tokens = simple_word_tokenize(text)
    verb_count = 0
    noun_count = 0

    for tok in tokens:
        analyses = analyzer.analyze(tok)
        if not analyses:
            continue
        pos = analyses[0].get("pos", "")

        if pos.startswith("verb"):
            verb_count += 1
        elif pos.startswith("noun"):
            noun_count += 1

    n = max(len(tokens), 1)
    return {
        "num_tokens": len(tokens),
        "num_verbs": verb_count,
        "num_nouns": noun_count,
        "verb_ratio": verb_count / n,
        "noun_ratio": noun_count / n,
    }


In [7]:
# CELL 4: Abstractness features (pattern-based proxy)

def abstractness_features(text):
    tokens = simple_word_tokenize(text)
    abstract_nouns = 0

    for tok in tokens:
        analyses = analyzer.analyze(tok)
        if not analyses:
            continue
        a = analyses[0]

        if a.get("pos", "").startswith("noun"):
            if a.get("pattern", "").startswith(("فِع", "فَع", "مَف")):
                abstract_nouns += 1

    return {
        "abstract_noun_ratio": abstract_nouns / max(len(tokens), 1)
    }


In [8]:
# CELL 5: Build verb-object frequency table

from collections import defaultdict

verb_object_counts = defaultdict(int)

for text in texts:
    tokens = simple_word_tokenize(text)
    for i in range(len(tokens) - 1):
        a1 = analyzer.analyze(tokens[i])
        a2 = analyzer.analyze(tokens[i + 1])
        if not a1 or not a2:
            continue

        if a1[0].get("ud") == "VERB" and a2[0].get("ud") == "NOUN":
            verb_object_counts[(tokens[i], tokens[i + 1])] += 1


In [9]:
# CELL 6: Selectional preference violation

def selectional_violation_feature(text):
    tokens = simple_word_tokenize(text)
    rare_pairs = 0
    total_pairs = 0

    for i in range(len(tokens) - 1):
        pair = (tokens[i], tokens[i + 1])
        if pair in verb_object_counts:
            total_pairs += 1
            if verb_object_counts[pair] <= 1:
                rare_pairs += 1

    return {
        "rare_verb_object_ratio": rare_pairs / max(total_pairs, 1)
    }


In [10]:
# CELL 7: Sensory verb + abstract noun clash

def sensory_mismatch_features(text):
    tokens = simple_word_tokenize(text)
    clashes = 0

    for i in range(len(tokens) - 1):
        v = analyzer.analyze(tokens[i])
        n = analyzer.analyze(tokens[i + 1])
        if not v or not n:
            continue

        if v[0].get("ud") == "VERB" and n[0].get("ud") == "NOUN":
            if n[0].get("pattern", "").startswith(("فِع", "فَع")):
                clashes += 1

    return {"sensory_abstract_clash": clashes}


In [11]:
# CELL 8: Agent–verb animacy clash (core majaz feature)

def agent_animacy_clash(text):
    tokens = simple_word_tokenize(text)
    clashes = 0

    for i in range(len(tokens) - 1):
        n = analyzer.analyze(tokens[i])
        v = analyzer.analyze(tokens[i + 1])
        if not n or not v:
            continue

        if n[0].get("ud") == "NOUN" and v[0].get("ud") == "VERB":
            if n[0].get("pattern", "").startswith(("فِع", "فَع", "مَف")):
                clashes += 1

    return {"agent_animacy_clash": clashes}


In [12]:
# CELL 9: Passive voice / agent suppression

def agent_suppression(text):
    tokens = simple_word_tokenize(text)
    passive = 0

    for tok in tokens:
        a = analyzer.analyze(tok)
        if a and a[0].get("vox") == "p":
            passive += 1

    return {"passive_ratio": passive / max(len(tokens), 1)}


In [13]:
# CELL 10: Root repetition density

def root_repetition(text):
    tokens = simple_word_tokenize(text)
    roots = []

    for tok in tokens:
        a = analyzer.analyze(tok)
        if a:
            roots.append(a[0].get("root"))

    return {
        "root_repetition_ratio": 1 - len(set(roots)) / max(len(roots), 1)
    }


In [14]:
# CELL 11: Feature dataframe builder

def build_feature_df(texts, feature_fn):
    return pd.DataFrame([feature_fn(t) for t in texts])


In [15]:
# CELL 12: Build feature matrices

X_morph = build_feature_df(texts, morpho_features)
X_abs   = build_feature_df(texts, abstractness_features)
X_sel   = build_feature_df(texts, selectional_violation_feature)
X_sens  = build_feature_df(texts, sensory_mismatch_features)

X_roles = pd.concat([
    build_feature_df(texts, agent_animacy_clash),
    build_feature_df(texts, agent_suppression),
    build_feature_df(texts, root_repetition),
], axis=1)

X_all = pd.concat([X_morph, X_abs, X_sel, X_sens, X_roles], axis=1)

print(X_all.shape)


(1684, 11)


In [16]:
# CELL 13: Train & evaluate classifier

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

def train_eval(X, y, name):
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    clf = LogisticRegression(max_iter=2000)
    clf.fit(Xtr, ytr)
    ypred = clf.predict(Xte)

    print(f"\n=== {name} ===")
    print(classification_report(yte, ypred, digits=3))


In [17]:
# CELL 14: Experiments

train_eval(X_morph, y, "Morpho-syntactic only")



=== Morpho-syntactic only ===
              precision    recall  f1-score   support

           0      0.569     0.439     0.496       253
           1      0.543     0.668     0.599       253

    accuracy                          0.553       506
   macro avg      0.556     0.553     0.547       506
weighted avg      0.556     0.553     0.547       506



In [18]:

train_eval(X_abs,   y, "Abstractness only")


=== Abstractness only ===
              precision    recall  f1-score   support

           0      0.500     1.000     0.667       253
           1      0.000     0.000     0.000       253

    accuracy                          0.500       506
   macro avg      0.250     0.500     0.333       506
weighted avg      0.250     0.500     0.333       506



c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

In [19]:

train_eval(X_sel,   y, "Selectional violation only")


=== Selectional violation only ===
              precision    recall  f1-score   support

           0      0.494     0.158     0.240       253
           1      0.499     0.838     0.625       253

    accuracy                          0.498       506
   macro avg      0.496     0.498     0.432       506
weighted avg      0.496     0.498     0.432       506



In [20]:

train_eval(X_sens,  y, "Sensory mismatch only")


=== Sensory mismatch only ===
              precision    recall  f1-score   support

           0      0.500     1.000     0.667       253
           1      0.000     0.000     0.000       253

    accuracy                          0.500       506
   macro avg      0.250     0.500     0.333       506
weighted avg      0.250     0.500     0.333       506



c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USER\miniconda3\envs\torch-gpu\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

In [21]:

train_eval(X_roles, y, "Role-based features only")


=== Role-based features only ===
              precision    recall  f1-score   support

           0      0.529     0.502     0.515       253
           1      0.526     0.553     0.539       253

    accuracy                          0.528       506
   macro avg      0.528     0.528     0.527       506
weighted avg      0.528     0.528     0.527       506



In [22]:

train_eval(X_all,   y, "ALL FEATURES COMBINED")


=== ALL FEATURES COMBINED ===
              precision    recall  f1-score   support

           0      0.563     0.458     0.505       253
           1      0.543     0.644     0.590       253

    accuracy                          0.551       506
   macro avg      0.553     0.551     0.547       506
weighted avg      0.553     0.551     0.547       506

